In [0]:
CREATE OR REPLACE VIEW w04 AS
WITH member_cohort AS (
  SELECT
    member_id,
    CONCAT(YEAR(MIN(effective_from)), '-Q', QUARTER(MIN(effective_till))) AS cohort_quarter
  FROM dim_plan
  GROUP BY member_id
),
cohort_claims AS (
  SELECT
    mc.cohort_quarter,
    f.claim_id,
    f.member_sk
  FROM member_cohort mc
  JOIN dim_member dm ON mc.member_id = dm.member_id
  JOIN fact_claims f ON f.member_sk = dm.member_sk
),
agg AS (
  SELECT
    mc.cohort_quarter,
    COUNT(DISTINCT mc.member_id) AS members,
    COUNT(DISTINCT cc.member_sk) AS claimants,
    COUNT(cc.claim_id) AS claim_count
  FROM member_cohort mc
  LEFT JOIN cohort_claims cc ON mc.cohort_quarter = cc.cohort_quarter
  GROUP BY mc.cohort_quarter
)
SELECT
  cohort_quarter,
  members,
  claimants,
  claim_count,
  ROUND(claim_count * 1.0 / NULLIF(members, 0), 4) AS claims_per_member,
  SUM(members) OVER (ORDER BY cohort_quarter ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_members
FROM agg
ORDER BY cohort_quarter;